# CyberSentinel-EU Stage 3 Part 3: Formal RAG Evaluation (RQ3)
**Fatema Husain Hasan (202508958) MSc AI Thesis**

Runs a frozen 15-question benchmark through BOTH systems (RAG and BM25),
saves all answers to a CSV for manual scoring on four dimensions:
correctness, faithfulness, citation validity, retrieval recall (+ refusal).

**Setup:** paste Azure key in Cell 2, then Runtime Run all (~5 min).

## 1. Setup

In [ ]:
!pip install openai chromadb rank-bm25 --quiet
import pandas as pd
import numpy as np
import json
import os
import textwrap
import zipfile
from openai import AzureOpenAI
import chromadb
print('ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 2. Azure credentials

In [ ]:
AZURE_ENDPOINT = "https://cybersentinel-resource.openai.azure.com/"
AZURE_KEY = "PASTE_YOUR_KEY"
EMBED_DEPLOYMENT = "text-embedding-3-small"
CHAT_DEPLOYMENT  = "gpt-5-mini"
API_VERSION = "2024-10-21"
client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_KEY, api_version=API_VERSION)
print("Azure connected")

Azure connected


## 3. Load vector store + corpus + BM25

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/Fatimaxx24/202508958_IT9099_Thesis/main/"
if not os.path.exists('chroma_gdpr'):
    !wget -q https://github.com/Fatimaxx24/202508958_IT9099_Thesis/raw/main/chroma_gdpr.zip
    with zipfile.ZipFile('chroma_gdpr.zip','r') as z: z.extractall('.')
chroma = chromadb.PersistentClient(path="./chroma_gdpr")
col = chroma.get_collection("gdpr_enforcement")

df = pd.read_csv(REPO_RAW + "gdpr_enforcement_tracker_full.csv", low_memory=False)
df['Summary'] = df['Summary'].astype(str)
df = df[df['Summary'].str.strip().str.len() > 20].reset_index(drop=True)

from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([t.lower().split() for t in df['Summary']])
print('vector store:', col.count(), '| BM25 index:', len(df))

vector store: 3201 | BM25 index: 3201


## 4. The frozen benchmark (15 questions, 6 types)

In [ ]:
BENCHMARK = [
 (1,  "single_record",  "What happened in the Belgian DPA's case against a hospital in 2024, and what was the fine?"),
 (2,  "single_record",  "Why did the Irish DPA fine Centric Health Ltd?"),
 (3,  "single_record",  "What security failure led to the Italian fine against Asl Napoli 3 Sud?"),
 (4,  "multi_record",   "What ransomware attacks affected healthcare organisations, and how did they occur?"),
 (5,  "multi_record",   "How do misdirected-email breaches differ from hacking incidents in these records?"),
 (6,  "multi_record",   "Compare security failures in the public sector with those in finance."),
 (7,  "country_agg",    "Which countries appear most often in cases involving insufficient security measures?"),
 (8,  "country_agg",    "What types of incidents did the Spanish DPA penalise?"),
 (9,  "sector_pattern", "What kinds of incidents most commonly affect the healthcare sector?"),
 (10, "sector_pattern", "What security incidents appear in the telecommunications sector?"),
 (11, "unanswerable",   "What was the average ransom amount paid?"),
 (12, "unanswerable",   "How many organisations reported these breaches to law enforcement?"),
 (13, "unanswerable",   "What was the total cost of recovery for these incidents?"),
 (14, "adversarial",    "Why did the UK ICO fine Google 50 million euros for a ransomware attack?"),
 (15, "adversarial",    "Which German company received the largest fine for a phishing attack in 2018?"),
]
print(f'{len(BENCHMARK)} questions frozen')
pd.DataFrame(BENCHMARK, columns=['id','type','question'])

15 questions frozen


,id,type,question
0,1,single_record,What happened in the Belgian DPA's case agains...
1,2,single_record,Why did the Irish DPA fine Centric Health Ltd?
2,3,single_record,What security failure led to the Italian fine ...
3,4,multi_record,What ransomware attacks affected healthcare or...
4,5,multi_record,How do misdirected-email breaches differ from ...
5,6,multi_record,Compare security failures in the public sector...
6,7,country_agg,Which countries appear most often in cases inv...
7,8,country_agg,What types of incidents did the Spanish DPA pe...
8,9,sector_pattern,What kinds of incidents most commonly affect t...
9,10,sector_pattern,What security incidents appear in the telecomm...


## 5. The two systems

In [ ]:
SYSTEM_PROMPT = """You are CyberSentinel-EU, an assistant that answers questions about GDPR
enforcement decisions using ONLY the retrieved records provided.

Rules:
1. Answer ONLY from the retrieved records. Never use outside knowledge.
2. Cite every claim with the record ID in square brackets, e.g. [ETid 2521].
3. If the retrieved records do not contain the answer, reply exactly:
   "Insufficient evidence in the corpus to answer this question."
4. If the question contains a false premise, correct it based on the records.
5. Be concise and factual. State uncertainty where it exists.
"""

def retrieve_rag(q, k=5):
    qv = client.embeddings.create(model=EMBED_DEPLOYMENT, input=q).data[0].embedding
    r = col.query(query_embeddings=[qv], n_results=k)
    return [{'ETid': m['ETid'], 'country': m['country'], 'sector': m['sector'],
             'score': round(1-d,3), 'text': doc}
            for doc, m, d in zip(r['documents'][0], r['metadatas'][0], r['distances'][0])]

def retrieve_bm25(q, k=5):
    s = bm25.get_scores(q.lower().split())
    top = np.argsort(s)[::-1][:k]
    return [{'ETid': str(df.iloc[i]['ETid']), 'country': str(df.iloc[i]['Country']),
             'sector': str(df.iloc[i]['Sector']), 'score': round(float(s[i]),3),
             'text': df.iloc[i]['Summary']} for i in top]

def generate(q, hits):
    ctx = "\n\n".join(f"[ETid {h['ETid']}] ({h['country']}, {h['sector']})\n{h['text']}" for h in hits)
    r = client.chat.completions.create(model=CHAT_DEPLOYMENT,
        messages=[{"role":"system","content":SYSTEM_PROMPT},
                  {"role":"user","content":f"Retrieved records:\n\n{ctx}\n\nQuestion: {q}"}])
    return r.choices[0].message.content
print('systems ready')

systems ready


## 6. Run the full benchmark through both systems

In [ ]:
rows = []
for qid, qtype, q in BENCHMARK:
    for system, retriever in [('RAG', retrieve_rag), ('BM25', retrieve_bm25)]:
        hits = retriever(q, 5)
        ans = generate(q, hits)
        rows.append({
            'qid': qid, 'type': qtype, 'system': system, 'question': q,
            'answer': ans,
            'retrieved_ETids': '; '.join(h['ETid'] for h in hits),
            'top_score': hits[0]['score'],
            'refused': 'Insufficient evidence' in ans,
            # ---- MANUAL SCORING COLUMNS (fill these in yourself) ----
            'correctness_0_2': '', 'faithfulness_0_2': '',
            'citation_validity_0_2': '', 'retrieval_recall_0_2': '', 'notes': '',
        })
    print(f'Q{qid} done ({qtype})')

results = pd.DataFrame(rows)
results.to_csv('rag_benchmark_results.csv', index=False, encoding='utf-8-sig')
print(f'\nSaved rag_benchmark_results.csv — {len(results)} rows (15 questions x 2 systems)')

Q1 done (single_record)
Q2 done (single_record)
Q3 done (single_record)
Q4 done (multi_record)
Q5 done (multi_record)
Q6 done (multi_record)
Q7 done (country_agg)
Q8 done (country_agg)
Q9 done (sector_pattern)
Q10 done (sector_pattern)
Q11 done (unanswerable)
Q12 done (unanswerable)
Q13 done (unanswerable)
Q14 done (adversarial)
Q15 done (adversarial)

Saved rag_benchmark_results.csv — 30 rows (15 questions x 2 systems)


## 7. Quick automatic summary (refusal behaviour)

In [ ]:
print("- REFUSAL BEHAVIOUR -")
unans = results[results['type']=='unanswerable']
for s in ['RAG','BM25']:
    sub = unans[unans.system==s]
    print(f'{s}: refused {sub.refused.sum()}/{len(sub)} unanswerable questions')

print("\n- ANSWERABLE QUESTIONS WRONGLY REFUSED -")
ans_q = results[results['type']!='unanswerable']
for s in ['RAG','BM25']:
    sub = ans_q[ans_q.system==s]
    print(f'{s}: refused {sub.refused.sum()}/{len(sub)} answerable questions')

print("\n- RETRIEVAL OVERLAP (how different are the two systems?) -")
for qid in range(1,16):
    r = set(results[(results.qid==qid)&(results.system=='RAG')]['retrieved_ETids'].iloc[0].split('; '))
    b = set(results[(results.qid==qid)&(results.system=='BM25')]['retrieved_ETids'].iloc[0].split('; '))
    print(f'Q{qid}: {len(r & b)}/5 records in common')

- REFUSAL BEHAVIOUR -
RAG: refused 3/3 unanswerable questions
BM25: refused 3/3 unanswerable questions

- ANSWERABLE QUESTIONS WRONGLY REFUSED -
RAG: refused 1/12 answerable questions
BM25: refused 3/12 answerable questions

- RETRIEVAL OVERLAP (how different are the two systems?) -
Q1: 1/5 records in common
Q2: 2/5 records in common
Q3: 2/5 records in common
Q4: 1/5 records in common
Q5: 0/5 records in common
Q6: 0/5 records in common
Q7: 0/5 records in common
Q8: 0/5 records in common
Q9: 0/5 records in common
Q10: 0/5 records in common
Q11: 1/5 records in common
Q12: 1/5 records in common
Q13: 0/5 records in common
Q14: 2/5 records in common
Q15: 0/5 records in common


## 8. Print all answers for manual scoring

In [ ]:
for qid, qtype, q in BENCHMARK:
    print("="*95)
    print(f"Q{qid} [{qtype}]  {q}")
    print("="*95)
    for s in ['RAG','BM25']:
        row = results[(results.qid==qid)&(results.system==s)].iloc[0]
        print(f"\n--- {s} ---")
        print(textwrap.fill(row['answer'], 92))
        print(f"retrieved: {row['retrieved_ETids']}")
    print()

Q1 [single_record]  What happened in the Belgian DPA's case against a hospital in 2024, and what was the fine?

--- RAG ---
The Belgian DPA fined a hospital EUR 200,000 after a ransomware attack via a server
vulnerability paralysed parts of its computer system and affected about 300,000 individuals;
the DPA found the hospital had not carried out a data protection impact assessment, lacked
an adequate information‑security policy, and failed to implement appropriate technical and
organisational measures (e.g., employee training and a process for security updates) [ETid
2521]. The retrieved record does not state the year of the decision, so I cannot confirm it
occurred in 2024 [ETid 2521].
retrieved: 2521; 1438; 63; 1905; 500

--- BM25 ---
The retrieved record states that the hospital suffered a ransomware attack exploiting a
server vulnerability that paralysed parts of its computer system and affected about 300,000
individuals; the Belgian DPA found the hospital had not carried out a dat

## 9. Next: score them
Download `rag_benchmark_results.csv` and fill the four scoring columns for each row:

**correctness** (0-2): 0 = wrong, 1 = partially correct, 2 = correct
**faithfulness** (0-2): 0 = claims unsupported by retrieved records, 1 = mostly supported, 2 = fully supported
**citation_validity** (0-2): 0 = citations wrong/missing, 1 = some valid, 2 = all citations support their claims
**retrieval_recall** (0-2): 0 = no relevant records retrieved, 1 = some, 2 = all relevant records retrieved

For unanswerable questions: correctness 2 = correctly refused, 0 = fabricated an answer.
For adversarial: correctness 2 = corrected the false premise.

Then re-upload the scored CSV and we compute the final RQ3 comparison.